# Graphing the Data
## Imports

In [65]:
import pandas as pd
import numpy as np
from currency_converter import CurrencyConverter, ECB_URL

# Needed for Plotly
import plotly.express as px
import plotly.tools as tools
import plotly.graph_objects as go

# Needed for Seaborn
# import seaborn as sns
# import matplotlib.pyplot as plt
# import matplotlib.ticker as ticker

## Run helpers and read in data

In [66]:
%run helpers.ipynb

In [67]:
oscar_with_imdb, imdb_non_oscar = read_data('oscar_with_imdb.csv', 'imdb_non_oscar.csv')

To plot all films at once regardless of nomination status, we use the columns as series to concatenate them into a data frame. This allows the budget and box office variables to be used as combined axes for Plotly Express.

In [68]:
oscar_with_imdb['Status'] = oscar_with_imdb['Winner'].apply(lambda row: 'Winner' if row == True else 'Nominated')
imdb_non_oscar['Status'] = 'No nomination'

budget_nom = oscar_with_imdb['budget']
gross_nom = oscar_with_imdb['gross_worldwide']

budget_imdb = imdb_non_oscar['budget']
gross_imdb = imdb_non_oscar['gross_worldwide']

oscar_noms = pd.concat([budget_nom, gross_nom, oscar_with_imdb['Status']], axis=1)
non_noms = pd.concat([budget_imdb, gross_imdb, imdb_non_oscar['Status']], axis=1)

combined_films = pd.concat([oscar_noms, non_noms], axis=0)
combined_films.dropna(how='any', inplace=True)

In [69]:
combined_films.shape

(14072, 3)

In [70]:
temp = oscar_with_imdb[oscar_with_imdb['Status'] == 'Nominated']

In [71]:
temp

,Unnamed: 0,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,...,opening_weekend_gross,gross_worldwide,gross_us_canada,release_date,countries_origin,production_companies,genres,languages,_merge,Status
0,0,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Noose,tt0019217,Richard Barthelmess,Richard Barthelmess,...,NaN,NaN,NaN,1928-01-29,['United States'],['First National Pictures'],['Drama'],"['None', 'English']",both,Nominated
1,1,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Patent Leather Kid,tt0018253,Richard Barthelmess,Richard Barthelmess,...,NaN,NaN,NaN,1927-09-01,['United States'],['First National Pictures'],"['Boxing', 'Drama', 'Romance', 'Sport', 'War']","['None', 'English']",both,Nominated
4,4,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,...,NaN,NaN,NaN,1928-06-04,['United States'],['DeMille Pictures Corporation'],['Drama'],['None'],both,Nominated
8,8,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,Sadie Thompson,tt0019344,Gloria Swanson,Gloria Swanson,...,NaN,NaN,NaN,1928-01-07,['United States'],['Gloria Swanson Pictures'],['Drama'],['English'],both,Nominated
9,9,1,1928,Production,ART DIRECTION,ART DIRECTION,Sunrise,tt0018455,Rochus Gliese,Rochus Gliese,...,NaN,"$121,848",NaN,1927-11-04,['United States'],['Fox Film Corporation'],"['Dark Romance', 'Psychological Drama', 'Drama...","['None', 'English']",both,Nominated
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9067,9067,97,2024,Writing,WRITING (Adapted Screenplay),WRITING (Adapted Screenplay),Sing Sing,tt28479262,"Screenplay by Clint Bentley, Greg Kwedar; Stor...","Clint Bentley, Greg Kwedar, Clarence Maclin, J...",...,"$137,119","$5,281,415","$3,110,476",2024-08-02,['United States'],"['Black Bear', 'Edith Productions', 'Marfa Pea...",['Drama'],['English'],both,Nominated
9069,9069,97,2024,Writing,WRITING (Original Screenplay),WRITING (Original Screenplay),The Brutalist,tt8999762,"Written by Brady Corbet, Mona Fastvold","Brady Corbet, Mona Fastvold",...,"$266,791","$48,520,129","$16,279,129",2025-01-24,"['United States', 'United Kingdom', 'Canada']","['Brookstreet Pictures', 'Kaplan Morrison', 'I...","['Epic', 'Drama']","['English', 'Hungarian', 'Italian', 'Hebrew', ...",both,Nominated
9070,9070,97,2024,Writing,WRITING (Original Screenplay),WRITING (Original Screenplay),A Real Pain,tt21823606,Written by Jesse Eisenberg,Jesse Eisenberg,...,"$228,856","$24,500,173","$8,344,978",2024-11-01,"['United States', 'Poland']","['Topic Studios', 'Extreme Emotions', 'Fruit T...","['Buddy Comedy', 'Comedy', 'Drama']","['English', 'Polish']",both,Nominated
9071,9071,97,2024,Writing,WRITING (Original Screenplay),WRITING (Original Screenplay),September 5,tt28082769,"Written by Moritz Binder, Tim Fehlbaum; Co-Wri...","Moritz Binder, Tim Fehlbaum, Alex David",...,"$80,802","$8,041,940","$2,508,723",2025-01-17,"['Germany', 'United States']","['BerghausWöbke Filmproduktion', 'Projected Pi...","['Docudrama', 'Period Drama', 'Tragedy', 'Dram...","['English', 'German', 'Hebrew']",both,Nominated


## Setting up data for scatter plot (Budget vs Box Office)

We want to categorize films by their academy award status or if they were not nominated. We begin by adding False entries to the `Winner` field in the joined data frame for Academy Award films, and then dropping any NA values in both joined data frames.

### Create `Status` field

In [72]:
temp_oscar_with_imdb = oscar_with_imdb.dropna(subset=['budget', 'gross_worldwide'])
temp_imdb_non_oscar = imdb_non_oscar.dropna(subset=['budget', 'gross_worldwide'])

# Nominess are marked as NaN so we replace with False
temp_oscar_with_imdb['Winner'] = oscar_with_imdb['Winner'].fillna(False)

In [73]:
temp_oscar_with_imdb['Status'] = temp_oscar_with_imdb['Winner'].apply(lambda row: 'Winner' if row else 'Nominated')

winners = temp_oscar_with_imdb[temp_oscar_with_imdb['Status'] == 'Winner']
nominees = temp_oscar_with_imdb[temp_oscar_with_imdb['Status'] == 'Nominated']
non_nom = temp_imdb_non_oscar

all_films = [winners, nominees, non_nom]
to_graph = [None, None, None]
film_groups = ['Won', 'Nominated', 'Not Nominated']

### Round budget and box and budget

Round everything and add to graph

In [74]:
print([film.shape for film in all_films])

[(1321, 29), (3993, 29), (8758, 20)]


In [75]:
## Initialize currency converter to pass into lambda

In [76]:
currency_conv = CurrencyConverter('eurofxref-hist.csv', fallback_on_missing_rate=True, fallback_on_wrong_date=True)

In [77]:
%run helpers.ipynb

In [78]:
# For each data frame, convert the currencies and add a new column

In [79]:
for i, df in enumerate(all_films):
    # ToDo check round_value function
    for j, row in df.iterrows():
        release_date = row['release_date']
    df['normalized_budget'] = df['budget'].apply(lambda x: normalize_budget(x, release_date, currency_conv))
        
    # df['adjusted_budget'] = df.apply(normalize_budget, axis=1)
    
    # df['rounded_budget'] = df['rounded_budget']
    # df['rounded_gross'] = np.log10(df['rounded_gross'])
    # to_graph[i] = df.groupby(['rounded_budget', 'rounded_gross']).size().reset_index(name='count')
    to_graph[i] = df

RUR8,000,000 Price(amount=Decimal('8000000'), currency=None) 8000000.0 

RUR8,000,000 Price(amount=Decimal('8000000'), currency=None) 8000000.0 

RUR1,000,000 Price(amount=Decimal('1000000'), currency=None) 1000000.0 

RUR1,000,000 Price(amount=Decimal('1000000'), currency=None) 1000000.0 

RUR622,000 Price(amount=Decimal('622000'), currency=None) 622000.0 

RUR1,000,000 Price(amount=Decimal('1000000'), currency=None) 1000000.0 

TRL1,200,000 Price(amount=Decimal('1200000'), currency=None) 1200000.0 

VEB4,273,248 Price(amount=Decimal('4273248'), currency=None) 4273248.0 

RUR25,000,000 Price(amount=Decimal('25000000'), currency=None) 25000000.0 

RUR46,300,000 Price(amount=Decimal('46300000'), currency=None) 46300000.0 

RUR650,000,000 Price(amount=Decimal('650000000'), currency=None) 650000000.0 

RUR1,000,000 Price(amount=Decimal('1000000'), currency=None) 1000000.0 

RUR190,000,000 Price(amount=Decimal('190000000'), currency=None) 190000000.0 

RUR150,000,000 Price(amount=Decimal('

In [64]:
[(1321, 31), (3993, 31), (8758, 20)]

[(1321, 31), (3993, 31), (8758, 20)]

In [44]:
for i, df in enumerate(to_graph):
    df = df[df['normalized_budget'] != 0]
    to_graph[i] = df

In [45]:
df1, df2, df3 = to_graph

In [46]:
all_films_scatter = pd.concat([df1, df2, df3], ignore_index=True)

In [47]:
all_films_scatter.shape

(13913, 32)

The `CurrencyConverter` library allowed for quick parsing, but had some failed lookups for currency. We have to drop films with currencies that did not properly convert or hard code the conversion rate. 

In [48]:
%run helpers.ipynb

In [22]:
for df in to_graph:
    df['rounded_budget'] = df['normalized_budget'].copy().apply(round_value)
    df['rounded_gross'] = df['gross_worldwide'].apply(round_value)
    
    # Drop NA and zero values
    df = df.dropna(subset=['rounded_budget', 'rounded_gross'])
    df = df[(df['rounded_budget'] > 0) & (df['rounded_gross'] > 0)]

In [23]:
print([film.shape for film in to_graph])

[(1315, 32), (3964, 32), (8634, 23)]


### Iterate through rows and add points to scatter plot

In [24]:
all_films_scatter['budget']

0            $200,000 (estimated)
1            $200,000 (estimated)
2          $2,000,000 (estimated)
3          $2,000,000 (estimated)
4            $200,000 (estimated)
                   ...           
13908      €5,000,000 (estimated)
13909     $45,000,000 (estimated)
13910    $250,000,000 (estimated)
13911     $14,500,000 (estimated)
13912      $8,000,000 (estimated)
Name: budget, Length: 13913, dtype: str

In [25]:
all_films_scatter['Status'].map(repr).value_counts()

Status
'No nomination'    8634
'Nominated'        3964
'Winner'           1315
Name: count, dtype: int64

In [26]:
# x = combined_films['budget']
# y = combined_films['gross_worldwide']
df1 = to_graph[0]
df2 = to_graph[1]
df3 = to_graph[2]
df3 = df3.rename(columns={'title': 'Film'})

all_films_scatter = pd.concat([df1, df2, df3], ignore_index=True)

# remove duplicate films between Oscar nominees/winners (every film should appear only once)
winner_ids = all_films_scatter.loc[all_films_scatter['Status'] == 'Winner', 'FilmId']
all_films_scatter = all_films_scatter[
    (all_films_scatter['Status'] != 'Nominated') |
    (~all_films_scatter['FilmId'].isin(winner_ids))
]

fig = px.scatter(
    all_films_scatter,
    x='rounded_budget',
    y='rounded_gross',
    size='votes',
    size_max=40,
    color='Status',
    title='Worldwide Box Office Earnings vs Film Budget',
    custom_data=['votes', 'rating', 'Film'],
    category_orders={
        'Status': ['No nomination', 'Nominated', 'Winner']
    },
    hover_data={
        'Status': False,
        'votes': ':,',
        'rating': True,
        'Film': True if 'Film' in all_films_scatter.columns else False
    },
    labels={
        'rounded_budget': 'Budget',
        'rounded_gross': 'Gross',
        'votes': 'User Votes',
        'rating': 'IMDb Rating'
    }
)

fig.update_layout(
    xaxis_type='log',
    xaxis_title='Budget (Log scaled)',
    yaxis_title='Worldwide Gross',
    xaxis_tickformat='$~s',
    yaxis_tickprefix='$',
    height=800
)

fig.update_traces(
    hovertemplate=
        "Budget=$%{x:.2~s}<br>"
        "Gross=$%{y:.2~s}<br>"
        "User Votes=%{customdata[0]:,}<br>"
        "IMDb Rating=%{customdata[1]}<br>"
        "Film=%{customdata[2]}"
        "<extra></extra>"
)

fig.show(config={'displayModeBar': False})

In [27]:
# ToDo: Add scatter for revenue and color code profitability

In [28]:
all_films_scatter.tail()

,Unnamed: 0,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,...,production_companies,genres,languages,_merge,Status,normalized_budget,rounded_budget,rounded_gross,Unnamed: 0.1,id
13908,59090,NaN,NaN,NaN,NaN,NaN,Muori di lei,NaN,NaN,NaN,...,"['Film House Bas Celik', 'Medusa Film', 'NIght...","['Comedy', 'Drama', 'Romance', 'Thriller']",['Italian'],NaN,No nomination,5548500.0,5548000.0,302000.0,59090.0,tt32478708
13909,59096,NaN,NaN,NaN,NaN,NaN,The Alto Knights,NaN,NaN,NaN,...,"['Warner Bros.', 'Winkler Films']","['Docudrama', 'Gangster', 'Period Drama', 'Tru...",['English'],NaN,No nomination,45000000.0,45000000.0,5853000.0,59096.0,tt21815562
13910,59104,NaN,NaN,NaN,NaN,NaN,Snow White,NaN,NaN,NaN,...,"['Walt Disney Pictures', 'Marc Platt Productio...","['Fairy Tale', 'Feel-Good Romance', 'Adventure...","['English', 'German']",NaN,No nomination,250000000.0,250000000.0,92629000.0,59104.0,tt6208148
13911,59119,NaN,NaN,NaN,NaN,NaN,Bagman,NaN,NaN,NaN,...,"['Temple Hill Entertainment', 'Media Capital T...",['Horror'],['English'],NaN,No nomination,14500000.0,14500000.0,1326000.0,59119.0,tt21201300
13912,59135,NaN,NaN,NaN,NaN,NaN,The Assessment,NaN,NaN,NaN,...,"['Augenschein Filmproduktion', 'Number 9 Films...","['Drama', 'Sci-Fi']",['English'],NaN,No nomination,8000000.0,8000000.0,153000.0,59135.0,tt32768323


### Create a profit column

In [ ]:
all_films_scatter['Profit'] = all_films_scatter['gross_worldwide'] - all_films_scatter['rounded_budget']

In [ ]:
# x = combined_films['budget']
# y = combined_films['gross_worldwide']
# df1 = to_graph[0]
# df2 = to_graph[1]
# df3 = to_graph[2]
# df3 = df3.rename(columns={'title': 'Film'})

# all_films_scatter = pd.concat([df1, df2, df3], ignore_index=True)
fig = px.scatter(all_films_scatter, x='rounded_budget', y='Profit', color='Status',
                hover_data={
                    'Film': True if 'Film' in all_films_scatter.columns else False
                }
        )
# fig = px.scatter(x, y, log_x=True, log_y=True)
# fig.update_layout(xaxis=dict(type='log'), xaxis_title='Budget (Log scaled)', yaxis_title='Worldwide Gross', title='Worldwide Box Office Earnings vs Film Budget')
# fig.update_layout(yaxis=dict(type='log'))
# fig.update_layout(yaxis=dict(type='log'))
fig.update_layout(height=600)
fig.show(config={'displayModeBar': False})

In [ ]:
profit_fig = px.scatter(
    all_films_scatter,
    x='budget',
    y='rounded_gross',
    size='votes',
    color='Status',
    hover_data=['Film'],
    labels={'rounded_budget': 'Budget', 'Profit': 'Profit', 'rounded_gross': 'Worldwide Gross'},
    title="Bubble Chart: Profit vs Budget vs Gross"
)
profit_fig.update_layout(height=700)
profit_fig.update_layout(xaxis=dict(type='log'), xaxis_title='Budget (Log scaled)', yaxis_title='Worldwide Gross', title='Worldwide Box Office Earnings vs Film Budget')
profit_fig.show()